# Test Text-Segmentierung Klassifkationsmodell

Training und Test eines ersten Klassifikationsodells zur Text-Segmentierung mit vorverarteiteten Dokumenten.

**Voraussetzungen**

Environment: model

Zur Erstellung des Datensatzes für das Training muss für die verwendeten PDF-Dokumente zurvor die Vorverarbeitung und Erstellung der Dataframes über das Notebook Preparation.ipynb ausgeführt werden. Für das Preprocessing wurde die Methode process_dataframe_basic aus dem Skript util/pdfprocessor.py genutzt.

**Datensatz**

Mit den aus den vorverarbeiteten PDF-Dokumenten erstellten und gelabelten Abschnitten wird für das Training der Klassifikationsmodelle unter Verwendung der baumbasierten Verfahren und Neuronalen Netzwerke ein Datensatz mit den in der folgenden Tabelle dargestellten Merkmalen erstellt.

Für jeden Abschnitt wird die Anzahl der Zeichen, Wörter sowie das Verhältnis der eindeutigen Wörter gegenüber der Gesamtzahl ermittelt und durch das Embedding-Modell ein Embedding-Vektoren erzeugt.

| Merkmal | Datentyp | Bedeutung |
| - | - | - |
| text_len | int | Anzahl Zeichen des Blocks |
| word_count | int | Anzahl Wörter des Blocks, unterteilt nach Leerzeichen |
| word_distinct_count_ratio | float | Verhältnis eindeutiger Wörter/Gesamtzahl |
| embedding_vektor | list[float] | Embedding-Vektor des Blocks |

**Versuchsaufbau**

Für eine Stichprobe der in einem einfachen Vorverarbeitungsschritt vorbereiteten Dokumente wird eine Cross-Validation mittels der Verfahren Random Forest, LGBM, XGBoost durchgeführt. Die Ziehung der Stichprobe erfolgt aus allen 1000 Dokumenten sowie aus den 867 Dokumenten nach Abgleich der Grenzen.

Für das Training werden in mehreren Durchläufen die gebildeten Merkmale der Abschnitte sowie in Kombination mit dem jeweils vorhergehenden und/oder nachfolgenden Abschnitt verwendet.

## Setup

### Pakete laden

In [1]:
import config

import time

import regex as re

import pandas as pd
import numpy as np

from openai import OpenAI

# Scikit learn
#from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, recall_score

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# Optimization
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real

from util.datasetprocessor import DatasetProcessor
from util.pdfprocessor import PDFProcessor
from util.px import PX
from util.helper import Helper

# Plot
import plotly.express as px
import plotly.graph_objects as go

### Daten laden

In [2]:
# PDF URLs
pdf_urls = pd.read_json(f'{config.dataset["path"]}pdf_urls.json', orient='index', dtype_backend='numpy_nullable').rename(columns={0: "pdf_urls"})

# PDF URLs clean
#pdf_urls_clean = pd.read_json(f'{config.dataset["path"]}pdf_urls_clean.json', orient='index', dtype_backend='numpy_nullable')#.rename(columns={0: "pdf_urls"})
#pdf_urls = pdf_urls_clean.loc[pdf_urls_clean.values]

### Datensatz erstellen

In [3]:
df = DatasetProcessor.dataset_create(
    config.dataset['path'], 
    pdf_urls.sample(n=round(pdf_urls.shape[0] * 0.2), axis=0, random_state=42),
    embedding_dimension=768
)#.sample(frac=1, axis=0, random_state=42)

2412.06193v1
2401.17469v2
2412.04943v2
2408.14507v2
2412.10128v2
2410.09555v2
2403.15810v1
2412.21038v2
2410.03667v5
2408.16309v2
2405.19803v2
2408.14872v2
2412.03935v2
2410.05184v2
2406.05923v2
2405.13964v4
2408.03431v3
2407.18271v4
2403.13015v2
2407.00037v2
2407.13389v3
2406.13580v2
2409.05222v1
2407.17037v2
2411.02770v3
2407.09978v4
2408.08408v3
2411.17165v2
2410.23106v2
2412.11926v2
2407.13314v3
2410.02927v1
2411.08777v2
2406.14350v1
2412.13519v1
2412.13914v3
2403.15628v3
2501.00398v2
2412.20294v2
2408.09648v4
2409.12004v2
2402.07135v2
2412.04168v2
2405.18220v2
2402.09721v6
2409.00095v1
2408.16245v3
2410.13100v1
2412.15929v3
2411.11402v4
2405.07102v3
2411.16264v1
2411.03319v1
2407.17151v2
2402.12350v3
2410.20081v3
2404.18137v2
2409.19777v2
2407.11518v1
2412.15726v3
2410.03816v2
2406.10612v1
2410.19555v1
2407.09638v2
2412.12495v1
2412.03606v1
2411.15684v3
2408.10561v3
2412.20019v2
2403.18329v4
2409.06325v3
2409.09862v3
2403.01012v4
2407.09321v3
2402.16724v1
2410.20812v3
2411.03955v1

### Pipeline erstellen

Pipeline für Preprocessing der Daten (Transformer) und Modell-Training erstellen.

### Preprocessing

Pipeline für Daten-Preprocessing erstellen.

#### Feature-Typen wählen

Unterteilung der Merkmale nach (Daten)Typ für Behandlung (Skalierung numerischer Werte oder Encoding kategorialer-/ordinaler Werte) durch Transformer innerhalb der Sklearn-Pipeline.

In [4]:
#numeric_features = ['text_len', 'word_count', 'word_distinct_count_ratio', 'text_len_prev', 'word_count_prev', 'word_distinct_count_ratio_prev', 'text_len_next', 'word_count_next', 'word_distinct_count_ratio_next']
numeric_features = df.drop(columns=['boundary']).columns.tolist()

#### Transformer

Transformer mit spezifischen Funktionen (Scaling, Encoding) für numerische, kategoriale und ordinale Merkmale erzeugen.

Zur Skalierung der nummerischen Merkmale wurden in einem Test mit dem RobustScaler die besten Ergebnisse erzielt.

In [5]:
numeric_transformer = Pipeline(steps=[
    # keine Durchführung von Imputation
    #('imputer', SimpleImputer(strategy='median')),#constant #mean
    # Werte skalieren
    ('scaler', RobustScaler())#StandardScaler, MinMaxScaler
])

#### Preprocessor

Preprocessor für Transformation der Merkmale entsprechend ihrem Typ mit dazugehörigem Transformer erzeugen.

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ]
)

#### Pipeline

Pipeline für Training des Modells mit Preprocessor und zu verwendendem Machine-Learning-Verfahren (Random Forest, LGBM oder XGBoost) erzeugen.

In [7]:
##model = RandomForestClassifier(random_state=42, n_jobs=8)
#model = XGBClassifier(random_state=42, n_jobs=8)
##model = LGBMClassifier(random_state=42, n_jobs=8)

#pipeline = Pipeline(steps=[
#    ('preprocessor', preprocessor), 
#    ('model', model)
#])

## Cross-Validation

Durchführung einer Cross-Validation.

Cross-Validation (Kreuzvalidierung) ist eine Methode, mit welcher geprüft werden kann, wie gut ein Modell auf unbekannten Daten funktioniert. Dabei wird der verwendete Datensatz in eine bestimmte Anzahl $n$ gleich großer Teile zerlegt (Splits) und das Modell abwechselnd mit $n-1$ Teilen trainiert und einem Teil getestet.

Das Ziel ist eine robustere Einschätzung der Fähigkeiten zur Generalisierung des Modells treffen zu können. Eine einzelne Aufteilung in Trainings- und Test-Daten kann zufällig zu einem besonders guten oder schlechten Ergebnis führen. Durch das abwechselnde Training und den Test kann ein stabileres Bild darüber erhalten werden, wie sich das Modell auf unbekannten Daten verhält.

Angewendet wird eine 5-Fold Bootstrap Cross-Validation (Stichproben aus Datensatz ziehen mit zurücklegen).

### Splits erstellen

Erstellung von 5 Splits, wechselseitiges Training mit 4 Splits und Test mit einem Split.

In [8]:
# Anzahl Splits
num_splits=5

# Splits erstellen
splits = DatasetProcessor.dataset_cross_validation(df, num_splits=num_splits)

In [9]:
#for i in range(len(splits)): print(splits[i][:10])

Für das Training der Modelle mit Verfahren des maschinellen Lernens wie Random Forest, LGBM und XGBoost wird das Paket sklearn verwendet. Mit Hilfe des ColumnTransformer wird eine Skalierung der nummerischen Werte (text_len, word_count, word_distinct_count_ratio sowie die Werte des Embedding-Vektors) unter Verwendung des RobustScalers durchgeführt. Da die Daten keine fehlenden Werte enthalten, wird keine Vervollständigung (Imputation) durchgeführt.

In [10]:
# Dictionary für Werte Modell, Split und Metriken
data = {'model': [], 'split': [], 'accuracy': [], 'recall': [], 'precision': []}#, 'time': []

# Classifier
classifier = [
    RandomForestClassifier,
    XGBClassifier,
    LGBMClassifier
]

# Merkmale
X = df.drop(columns=['boundary'])
# Zielvariable
y = df['boundary']

# Über Classifier iterieren
for c in range(len(classifier)):
    # Über Splits iterieren
    for i in range(num_splits):
        # Beobachtungen für Train-/Test-Split aus Datensatz selektieren
        X_train, X_test, y_train, y_test = X.loc[~splits[i]], X.loc[splits[i]], y.loc[~splits[i]], y.loc[splits[i]]
    
        # Modell erstellen
        model = (classifier[c])(random_state=42, n_jobs=8)
    
        # Pipeline erstellen
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor), 
            ('model', model)
        ])

        # Zeitstempel
        #time_start = time.time()
        
        # Training
        pipeline.fit(X_train, y_train)
        
        # Vorhersage
        y_pred = pipeline.predict(X_test)
        
        # Werte zu Listen hinzufügen
        data['model'].append((classifier[c]).__name__)
        data['split'].append(i)
        data['accuracy'].append(accuracy_score(y_test, y_pred))
        data['recall'].append(recall_score(y_test, y_pred))
        data['precision'].append(precision_score(y_test, y_pred))
        #data['time'].append(time.time() - time_start)

# Dataframe erstellen
df_cv = pd.DataFrame.from_dict(data)

[LightGBM] [Info] Number of positive: 2719, number of negative: 102167
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.442990 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 393201
[LightGBM] [Info] Number of data points in the train set: 104886, number of used features: 1542
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.025923 -> initscore=-3.626345
[LightGBM] [Info] Start training from score -3.626345


/home/chris/venv/model/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2746, number of negative: 102463
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.433147 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 393206
[LightGBM] [Info] Number of data points in the train set: 105209, number of used features: 1542
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.026100 -> initscore=-3.619356
[LightGBM] [Info] Start training from score -3.619356


/home/chris/venv/model/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2715, number of negative: 102347
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.425526 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 393208
[LightGBM] [Info] Number of data points in the train set: 105062, number of used features: 1542
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.025842 -> initscore=-3.629577
[LightGBM] [Info] Start training from score -3.629577


/home/chris/venv/model/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2752, number of negative: 102274
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.431729 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 393206
[LightGBM] [Info] Number of data points in the train set: 105026, number of used features: 1542
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.026203 -> initscore=-3.615328
[LightGBM] [Info] Start training from score -3.615328


/home/chris/venv/model/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2742, number of negative: 102346
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.444572 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 393204
[LightGBM] [Info] Number of data points in the train set: 105088, number of used features: 1542
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.026092 -> initscore=-3.619672
[LightGBM] [Info] Start training from score -3.619672


/home/chris/venv/model/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
# pdf_urls.json
# einfache Vorverarbeitung
# Embedding-Dimension: 768
# Datensatz-Anteil: 1000 * 0.2
# Embedding-Vektor aktueller Abschnitt

# Anzahl Dokumente
#print(round(pdf_urls.shape[0] * 0.2))
#200
# Anzahl Abschnitte
#print(df.shape[0])
#131232
# Anzahl Grenzen
#print(df.loc[df['boundary'] == True].shape[0])
#3412

df_cv.groupby(by=['model'])[['accuracy', 'recall', 'precision']].agg(['mean', 'min', 'max', 'std'])

#accuracy	recall	precision
#mean	min	max	std	mean	min	max	std	mean	min	max	std
#model												
#LGBMClassifier	0.989841	0.989144	0.990470	0.000529	0.674153	0.645455	0.695195	0.019809	0.909562	0.896686	0.917625	0.007743
#RandomForestClassifier	0.981573	0.981363	0.981977	0.000242	0.288911	0.269697	0.308465	0.015461	0.994824	0.989362	1.000000	0.003779
#XGBClassifier	0.990542	0.990079	0.991097	0.000419	0.698854	0.674242	0.727403	0.020341	0.915159	0.908163	0.921818	0.006327

accuracy                                  recall  \
                            mean       min       max       std      mean   
model                                                                      
LGBMClassifier          0.989841  0.989144  0.990470  0.000529  0.674153   
RandomForestClassifier  0.981573  0.981363  0.981977  0.000242  0.288911   
XGBClassifier           0.990542  0.990079  0.991097  0.000419  0.698854   

                                                     precision            \
                             min       max       std      mean       min   
model                                                                      
LGBMClassifier          0.645455  0.695195  0.019809  0.909562  0.896686   
RandomForestClassifier  0.269697  0.308465  0.015461  0.994824  0.989362   
XGBClassifier           0.674242  0.727403  0.020341  0.915159  0.908163   

                                            
                             max       std  
model                                       
LGBMClassifier          0.917625  0.007743  
RandomForestClassifier  1.000000  0.003779  
XGBClassifier           0.921818  0.006327

In [11]:
# pdf_urls.json
# einfache Vorverarbeitung
# Embedding-Dimension: 768
# Datensatz-Anteil: 1000 * 0.2
# Embedding-Vektor aktueller+vorheriger Abschnitt

df_cv.groupby(by=['model'])[['accuracy', 'recall', 'precision']].agg(['mean', 'min', 'max', 'std'])

#accuracy	recall	precision
#mean	min	max	std	mean	min	max	std	mean	min	max	std
#model												
#LGBMClassifier	0.989725	0.989301	0.990432	0.000448	0.677608	0.667169	0.690265	0.008825	0.897331	0.880157	0.923077	0.016161
#RandomForestClassifier	0.980387	0.979613	0.980870	0.000517	0.237174	0.198795	0.250375	0.021562	0.997306	0.992481	1.000000	0.003731
#XGBClassifier	0.989748	0.989301	0.990585	0.000497	0.679374	0.670165	0.696165	0.010734	0.896370	0.881657	0.923679	0.017543

accuracy                                  recall  \
                            mean       min       max       std      mean   
model                                                                      
LGBMClassifier          0.989725  0.989301  0.990432  0.000448  0.677608   
RandomForestClassifier  0.980387  0.979613  0.980870  0.000517  0.237174   
XGBClassifier           0.989748  0.989301  0.990585  0.000497  0.679374   

                                                     precision            \
                             min       max       std      mean       min   
model                                                                      
LGBMClassifier          0.667169  0.690265  0.008825  0.897331  0.880157   
RandomForestClassifier  0.198795  0.250375  0.021562  0.997306  0.992481   
XGBClassifier           0.670165  0.696165  0.010734  0.896370  0.881657   

                                            
                             max       std  
model                                       
LGBMClassifier          0.923077  0.016161  
RandomForestClassifier  1.000000  0.003731  
XGBClassifier           0.923679  0.017543

In [12]:
# pdf_urls.json
# einfache Vorverarbeitung
# Embedding-Dimension: 768
# Datensatz-Anteil: 1000 * 0.2
# Embedding-Vektor aktueller+vorheriger+folgender Abschnitt

df_cv.groupby(by=['model'])[['accuracy', 'recall', 'precision']].agg(['mean', 'min', 'max', 'std'])

#accuracy	recall	precision
#mean	min	max	std	mean	min	max	std	mean	min	max	std
#model												
#LGBMClassifier	0.990518	0.990168	0.990839	0.000264	0.715800	0.701493	0.733333	0.013155	0.904080	0.893993	0.910853	0.006469
#RandomForestClassifier	0.979182	0.978309	0.980347	0.000782	0.210835	0.201493	0.219310	0.008354	0.993206	0.985915	1.000000	0.004995
#XGBClassifier	0.990196	0.989648	0.990743	0.000439	0.705849	0.690076	0.724138	0.014401	0.900498	0.883929	0.917969	0.012880

accuracy                                  recall  \
                            mean       min       max       std      mean   
model                                                                      
LGBMClassifier          0.990518  0.990168  0.990839  0.000264  0.715800   
RandomForestClassifier  0.979182  0.978309  0.980347  0.000782  0.210835   
XGBClassifier           0.990196  0.989648  0.990743  0.000439  0.705849   

                                                     precision            \
                             min       max       std      mean       min   
model                                                                      
LGBMClassifier          0.701493  0.733333  0.013155  0.904080  0.893993   
RandomForestClassifier  0.201493  0.219310  0.008354  0.993206  0.985915   
XGBClassifier           0.690076  0.724138  0.014401  0.900498  0.883929   

                                            
                             max       std  
model                                       
LGBMClassifier          0.910853  0.006469  
RandomForestClassifier  1.000000  0.004995  
XGBClassifier           0.917969  0.012880

In [11]:
# pdf_urls.json
# einfache Vorverarbeitung
# Embedding-Dimension: 768
# Datensatz-Anteil: 1000 * 0.2
# Embedding-Vektor aktueller+folgender Abschnitt

df_cv.groupby(by=['model'])[['accuracy', 'recall', 'precision']].agg(['mean', 'min', 'max', 'std'])

#accuracy	recall	precision
#mean	min	max	std	mean	min	max	std	mean	min	max	std
#model												
#LGBMClassifier	0.990436	0.989850	0.991085	0.000561	0.708433	0.675758	0.731343	0.020818	0.900618	0.886691	0.914122	0.011028
#RandomForestClassifier	0.980114	0.979807	0.980709	0.000359	0.231952	0.213636	0.249641	0.016202	0.995889	0.986486	1.000000	0.006077
#XGBClassifier	0.990893	0.989964	0.991670	0.000656	0.722302	0.681818	0.750359	0.025012	0.905964	0.891544	0.926606	0.017035

accuracy                                  recall  \
                            mean       min       max       std      mean   
model                                                                      
LGBMClassifier          0.990436  0.989850  0.991085  0.000561  0.708433   
RandomForestClassifier  0.980114  0.979807  0.980709  0.000359  0.231952   
XGBClassifier           0.990893  0.989964  0.991670  0.000656  0.722302   

                                                     precision            \
                             min       max       std      mean       min   
model                                                                      
LGBMClassifier          0.675758  0.731343  0.020818  0.900618  0.886691   
RandomForestClassifier  0.213636  0.249641  0.016202  0.995889  0.986486   
XGBClassifier           0.681818  0.750359  0.025012  0.905964  0.891544   

                                            
                             max       std  
model                                       
LGBMClassifier          0.914122  0.011028  
RandomForestClassifier  1.000000  0.006077  
XGBClassifier           0.926606  0.017035

Für die Stichprobe von 200 aus den insgesamt 1000 Dokumenten mit einfacher Vorverarbeitung wurde mit dem Verfahren XGBoost in der Cross-Validation mit einem durchschnittlichen Recall von 0.722 der größte Teil der Grenzen richtig klassifierziert, wobei die Werte mit einem Minimum von 0.682 und Maximum von 0.75 relativ stark schwanken. Der durchschnittliche Anteil der richtig positiv vorhergesagten Grenzen an der Gesamtzahl der positiv klassifizierten ist mit 0.987 für Random Forest am höchsten, für dieses Verfahren wurde mit einem Werte von 0.232 aber ein sehr geringer Recall erzielt.

In [17]:
# pdf_urls_clean.json
# einfache Vorverarbeitung
# Embedding-Dimension: 768
# Datensatz-Anteil: 867 * 0.2
# Embedding-Vektor aktueller+folgender Abschnitt

# Anzahl Dokumente
#print(round(pdf_urls.shape[0] * 0.2))
#173
# Anzahl Abschnitte
#print(df.shape[0])
#107612
# Anzahl Grenzen
#print(df.loc[df['boundary'] == True].shape[0])
#3404

df_cv.groupby(by=['model'])[['accuracy', 'recall', 'precision']].agg(['mean', 'min', 'max', 'std'])

#accuracy	recall	precision
#mean	min	max	std	mean	min	max	std	mean	min	max	std
#model												
#LGBMClassifier	0.990036	0.989571	0.990364	0.000393	0.741700	0.716895	0.759644	0.017762	0.925869	0.920863	0.931608	0.004577
#RandomForestClassifier	0.975906	0.974555	0.977105	0.000974	0.232586	0.207246	0.267564	0.022663	0.997500	0.993056	1.000000	0.003458
#XGBClassifier	0.990054	0.989460	0.991001	0.000679	0.737561	0.715373	0.765321	0.018446	0.930963	0.923650	0.943503	0.008045

accuracy                                  recall  \
                            mean       min       max       std      mean   
model                                                                      
LGBMClassifier          0.990036  0.989571  0.990364  0.000393  0.741700   
RandomForestClassifier  0.975906  0.974555  0.977105  0.000974  0.232586   
XGBClassifier           0.990054  0.989460  0.991001  0.000679  0.737561   

                                                     precision            \
                             min       max       std      mean       min   
model                                                                      
LGBMClassifier          0.716895  0.759644  0.017762  0.925869  0.920863   
RandomForestClassifier  0.207246  0.267564  0.022663  0.997500  0.993056   
XGBClassifier           0.715373  0.765321  0.018446  0.930963  0.923650   

                                            
                             max       std  
model                                       
LGBMClassifier          0.931608  0.004577  
RandomForestClassifier  1.000000  0.003458  
XGBClassifier           0.943503  0.008045

Bei Verwendung der Stichprobe von 173 aus den 867 abgeglichenen und einfach vorverarbeiteten Dokumenten wurde mit dem Verfahren LGBM der höchste Recall von 0.742 erzielt und das Vorhersageergebnis trotz Verwendung einer geringen Anzahl von Daten, 107612 anstelle von 131232 Abschnitten bei ungefähr gleicher Anzahl an Grenzen (3404 statt 3412), gegenüber dem Training mit allen 1000 Dokumenten leicht verbessert. Für das Verfahren XGBoost hat sich der Recall von 0.722 auf 0.738 sowie die Precision von 0.906 auf 0.931 leicht verbessert und die Schwankung zwischen den Minimal- und Maximal-Werten etwas verringert.

In [13]:
# pdf_urls_clean.json
# erweiterte Vorverarbeitung
# Embedding-Dimension: 768
# Datensatz-Anteil: 867 * 0.2
# Embedding-Vektor aktueller+folgender Abschnitt

# Anzahl Dokumente
#print(round(pdf_urls.shape[0] * 0.2))
#173
# Anzahl Abschnitte
#print(df.shape[0])
#56766
# Anzahl Grenzen
#print(df.loc[df['boundary'] == True].shape[0])
#3213

df_cv.groupby(by=['model'])[['accuracy', 'recall', 'precision']].agg(['mean', 'min', 'max', 'std'])

#accuracy	recall	precision
#mean	min	max	std	mean	min	max	std	mean	min	max	std
#model												
#LGBMClassifier	0.986329	0.985261	0.988263	0.001149	0.803486	0.794913	0.825472	0.012718	0.946173	0.936594	0.959781	0.009437
#RandomForestClassifier	0.966602	0.965301	0.968109	0.001326	0.411329	0.390750	0.447531	0.024057	0.992492	0.989761	0.996255	0.002355
#XGBClassifier	0.986558	0.985705	0.987999	0.000888	0.805886	0.786284	0.819182	0.012881	0.948273	0.937163	0.961255	0.009504

accuracy                                  recall  \
                            mean       min       max       std      mean   
model                                                                      
LGBMClassifier          0.986329  0.985261  0.988263  0.001149  0.803486   
RandomForestClassifier  0.966602  0.965301  0.968109  0.001326  0.411329   
XGBClassifier           0.986558  0.985705  0.987999  0.000888  0.805886   

                                                     precision            \
                             min       max       std      mean       min   
model                                                                      
LGBMClassifier          0.794913  0.825472  0.012718  0.946173  0.936594   
RandomForestClassifier  0.390750  0.447531  0.024057  0.992492  0.989761   
XGBClassifier           0.786284  0.819182  0.012881  0.948273  0.937163   

                                            
                             max       std  
model                                       
LGBMClassifier          0.959781  0.009437  
RandomForestClassifier  0.996255  0.002355  
XGBClassifier           0.961255  0.009504

Mit den 867 abgeglichenen und in einer erweiterten Vorverarbeitung behandelten Dokumenten konnte unter Verwendung der gleichen Stichprobe von 173 Dokumenten und einer wiederum geringeren Anzahl von Abschnitten (56766 statt 107612) sowie im Verhältnis höheren Anzahl an Grenzen (3213 statt 3404) das Vorhersageergebnis für alle Verfahren deutlich gesteigert werden. Unter Einsatz des Verfahrens XGBoost, hat sich der Recall gegenüber der Cross-Validation mit den einfach vorverarbeiteten Dokumenten von 0.738 auf 0.806 sowie die Precision von 0.930963 auf 0.948273 erhöht und die Schwankung zwischen den Ergebnissen der einzelnen Validierungsschritten weiter verringert. Insgesamt wurde mit dem Verfahren das beste Ergebnis erzielt und es wird für die weitere Optimierung und Vorhersage verwendet.